# 03 - Head-to-head with NVIDIA's Ising pre-decoder
Runs NVIDIA's real 3D-CNN pre-decoder + PyMatching, plain PyMatching, and **our** local pre-decoder on the *same* shots (NVIDIA's own circuit and noise model).

**One-time access step:** the Ising weights are gated on Hugging Face. Open https://huggingface.co/nvidia/ising_decoder_surface_code_1_fast , accept the terms, create a *read* token, and add it in Colab (key icon, left bar) as a secret named `HF_TOKEN` with notebook access ON. The token stays in your Colab secrets; never paste it into a cell or commit it. The weights are under the NVIDIA Open Model License and are downloaded at run time only.

In [ ]:
# ---- edit this: your public GitHub repo ----
GITHUB_REPO = "Rhytam23/ccn"
import os, subprocess, sys
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_REPO}.git", "/content/repo"], check=True)
%cd /content/repo
!pip -q install -r requirements-colab.txt
sys.path.insert(0, "/content/repo/src")  # editable installs are not visible to an already-running kernel
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # read from Colab secrets, never printed
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/Ising-Decoding.git", "third_party/Ising-Decoding"], check=True)
!pip -q install safetensors omegaconf hydra-core huggingface_hub beliefmatching

In [ ]:
!python scripts/run_ising_bench.py --repo third_party/Ising-Decoding --download --device cuda --distances 9 13 --ps 0.003 0.005 --shots 20000 --tag colab-ising

In [ ]:
!python scripts/make_report.py
!zip -qr results_ising.zip results/colab-ising docs/index.html
from google.colab import files; files.download('results_ising.zip')

Interpretation: compare LER (must match PyMatching), stage 1 vs stage 2 seconds, and total shots/s. Batch-1 latency is expected to favour plain PyMatching for NVIDIA's model too (see NVIDIA's own caveat in their cookbook).